# LLM Inference Benchmark Analysis

Analyzing performance across different LLM inference engines:
- Azure OpenAI (GPT-4o, GPT-4o-mini)
- Ollama (local Llama models)
- Databricks Foundation Models

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
# Load benchmark results
with open('../results/benchmark_results.json', 'r') as f:
    data = json.load(f)

# Convert to DataFrames
summaries_df = pd.DataFrame(data['summaries'])
raw_df = pd.DataFrame(data['raw_results'])

print(f"Benchmark run: {data['timestamp']}")
print(f"Total runs: {len(raw_df)}")
summaries_df

## 1. Tokens Per Second Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Create labels
summaries_df['label'] = summaries_df['engine'] + '\n' + summaries_df['model'].str.split('/').str[-1]

# Bar chart
bars = ax.bar(summaries_df['label'], summaries_df['avg_tps'], 
              yerr=summaries_df['std_tps'], capsize=5, color=sns.color_palette('husl', len(summaries_df)))

ax.set_ylabel('Tokens Per Second (TPS)', fontsize=12)
ax.set_xlabel('Engine / Model', fontsize=12)
ax.set_title('LLM Inference Speed Comparison', fontsize=14, fontweight='bold')

# Add value labels on bars
for bar, val in zip(bars, summaries_df['avg_tps']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f'{val:.1f}', ha='center', va='bottom', fontsize=10)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../results/tps_comparison.png', dpi=150)
plt.show()

## 2. Latency Comparison (TTFT & Total)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time to First Token
axes[0].barh(summaries_df['label'], summaries_df['avg_ttft_ms'], color='steelblue')
axes[0].set_xlabel('Time to First Token (ms)', fontsize=12)
axes[0].set_title('Time to First Token (TTFT)', fontsize=12, fontweight='bold')
for i, v in enumerate(summaries_df['avg_ttft_ms']):
    axes[0].text(v + 5, i, f'{v:.0f}ms', va='center')

# P95 Latency
axes[1].barh(summaries_df['label'], summaries_df['p95_latency_ms'], color='coral')
axes[1].set_xlabel('P95 Latency (ms)', fontsize=12)
axes[1].set_title('P95 Total Latency', fontsize=12, fontweight='bold')
for i, v in enumerate(summaries_df['p95_latency_ms']):
    axes[1].text(v + 5, i, f'{v:.0f}ms', va='center')

plt.tight_layout()
plt.savefig('../results/latency_comparison.png', dpi=150)
plt.show()

## 3. Cost Analysis

In [ ]:
# Calculate cost for 1M tokens
summaries_df['cost_per_1m'] = summaries_df['cost_per_1k_tokens'] * 1000

fig, ax = plt.subplots(figsize=(10, 6))

# Only show non-zero costs
cost_df = summaries_df[summaries_df['cost_per_1m'] > 0].copy()

if len(cost_df) > 0:
    bars = ax.bar(cost_df['label'], cost_df['cost_per_1m'], color='green', alpha=0.7)
    ax.set_ylabel('Cost per 1M Tokens ($)', fontsize=12)
    ax.set_xlabel('Engine / Model', fontsize=12)
    ax.set_title('Cost Comparison (Cloud APIs)', fontsize=14, fontweight='bold')
    
    for bar, val in zip(bars, cost_df['cost_per_1m']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                f'${val:.2f}', ha='center', va='bottom', fontsize=10)
    
    plt.xticks(rotation=45, ha='right')
else:
    ax.text(0.5, 0.5, 'All tested models are free (local inference)', 
            ha='center', va='center', fontsize=14, transform=ax.transAxes)

plt.tight_layout()
plt.savefig('../results/cost_comparison.png', dpi=150)
plt.show()

## 4. Cost vs Performance Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Scatter plot: TPS vs Cost, size = 1/latency (faster = bigger)
sizes = 1000 / (summaries_df['avg_latency_ms'] + 1)  # Inverse of latency

scatter = ax.scatter(
    summaries_df['cost_per_1k_tokens'], 
    summaries_df['avg_tps'],
    s=sizes,
    c=range(len(summaries_df)),
    cmap='husl',
    alpha=0.7,
    edgecolors='black'
)

# Add labels
for i, row in summaries_df.iterrows():
    ax.annotate(row['model'].split('/')[-1][:20], 
                (row['cost_per_1k_tokens'], row['avg_tps']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

ax.set_xlabel('Cost per 1K Tokens ($)', fontsize=12)
ax.set_ylabel('Tokens Per Second', fontsize=12)
ax.set_title('Cost vs Performance Trade-off\n(bubble size = speed)', fontsize=14, fontweight='bold')

# Add quadrant labels
ax.axhline(y=summaries_df['avg_tps'].median(), color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=summaries_df['cost_per_1k_tokens'].median(), color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../results/cost_performance_tradeoff.png', dpi=150)
plt.show()

## 5. Raw Results Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TPS Distribution
raw_df['label'] = raw_df['engine'] + ' / ' + raw_df['model'].str.split('/').str[-1].str[:15]
sns.boxplot(data=raw_df, x='label', y='tokens_per_second', ax=axes[0])
axes[0].set_title('Tokens Per Second Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

# Latency Distribution
sns.boxplot(data=raw_df, x='label', y='total_latency_ms', ax=axes[1])
axes[1].set_title('Latency Distribution (ms)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/distributions.png', dpi=150)
plt.show()

## 6. Summary Table

In [ ]:
# Create summary table for README
summary_table = summaries_df[['engine', 'model', 'quantization', 'avg_tps', 'avg_ttft_ms', 'p95_latency_ms', 'cost_per_1k_tokens']].copy()
summary_table.columns = ['Engine', 'Model', 'Quantization', 'Avg TPS', 'Avg TTFT (ms)', 'P95 Latency (ms)', 'Cost/1K Tokens']
summary_table['Avg TPS'] = summary_table['Avg TPS'].round(1)
summary_table['Avg TTFT (ms)'] = summary_table['Avg TTFT (ms)'].round(0)
summary_table['P95 Latency (ms)'] = summary_table['P95 Latency (ms)'].round(0)
summary_table['Cost/1K Tokens'] = summary_table['Cost/1K Tokens'].apply(lambda x: f'${x:.5f}' if x > 0 else 'Free')

print(summary_table.to_markdown(index=False))

## 7. Recommendations

In [ ]:
# Generate recommendations based on results
print("="*60)
print("RECOMMENDATIONS")
print("="*60)

# Fastest
fastest = summaries_df.loc[summaries_df['avg_tps'].idxmax()]
print(f"\n🚀 Fastest (TPS): {fastest['engine']} / {fastest['model']}")
print(f"   {fastest['avg_tps']:.1f} tokens/sec")

# Lowest latency
lowest_lat = summaries_df.loc[summaries_df['avg_ttft_ms'].idxmin()]
print(f"\n⚡ Lowest TTFT: {lowest_lat['engine']} / {lowest_lat['model']}")
print(f"   {lowest_lat['avg_ttft_ms']:.0f}ms time to first token")

# Best value (TPS per dollar, for non-free)
paid = summaries_df[summaries_df['cost_per_1k_tokens'] > 0].copy()
if len(paid) > 0:
    paid['value'] = paid['avg_tps'] / (paid['cost_per_1k_tokens'] * 1000 + 0.01)
    best_value = paid.loc[paid['value'].idxmax()]
    print(f"\n💰 Best Value (paid): {best_value['engine']} / {best_value['model']}")
    print(f"   {best_value['avg_tps']:.1f} TPS at ${best_value['cost_per_1k_tokens']:.5f}/1K tokens")

# Free option
free = summaries_df[summaries_df['cost_per_1k_tokens'] == 0]
if len(free) > 0:
    best_free = free.loc[free['avg_tps'].idxmax()]
    print(f"\n🆓 Best Free: {best_free['engine']} / {best_free['model']}")
    print(f"   {best_free['avg_tps']:.1f} TPS (local inference)")

print("\n" + "="*60)